In [1]:
import pickle, os, logging, warnings, sys
from pathlib import Path
import pandas as pd
from loguru import logger

logger.remove() 
logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

data_path = Path.cwd().parent / "data" / "final_dataset_sections.pickle"
with open(data_path, 'rb') as f:
    final_dataset_sections = pickle.load(f)

def get_patient_sections(dataset, patient_idx):
    return [''.join(sec) for sec in dataset['train'][patient_idx]['sections_tokens']]

In [ ]:
import spacy, medspacy, stanza, nltk
from nltk.tokenize import sent_tokenize

nlp_spacy = spacy.load('el_core_news_lg')
nlp_medspacy = medspacy.load()
nlp_stanza = stanza.Pipeline('el', processors='tokenize', verbose=False)

def engine_spacy(text):
    return [s.text.strip() for s in nlp_spacy(text).sents if s.text.strip()]

def engine_medspacy(text):
    return [s.text.strip().replace('\n', ' ') for s in nlp_medspacy(text).sents if s.text.strip()]

def engine_stanza(text):
    return [s.text.strip().replace('\n', ' ') for s in nlp_stanza(text).sentences if s.text.strip()]

def engine_nltk(text):
    return [s.strip() for s in sent_tokenize(text, language='greek') if s.strip()]

MODELS = {
    "spaCy (LG)": engine_spacy,
    "medspaCy": engine_medspacy,
    "Stanza": engine_stanza,
    "NLTK": engine_nltk
}

In [ ]:
def run_benchmarks(patient_list, sections_limit=None):
    all_results = []

    for p_idx in patient_list:
        sections = get_patient_sections(final_dataset_sections, p_idx)
        if sections_limit: sections = sections[:sections_limit]

        for s_idx, text in enumerate(sections):
            row = {"Patient": p_idx, "Section": s_idx}
            
            print(f"\n{'='*20} Testing Patient {p_idx} | Sec {s_idx} {'='*20}")
            
            for name, func in MODELS.items():
                sents = func(text)
                row[name] = len(sents) 
                
                print(f"\n[{name}] Found {len(sents)} sentences:")
                for i, s in enumerate(sents[:3], 1):
                    print(f"  {i}. {s[:80]}...") 
            
            all_results.append(row)
    
    return pd.DataFrame(all_results)

In [4]:
def compare_patient_output(patient_idx, section_idx):
    sections = get_patient_sections(final_dataset_sections, patient_idx)
    raw_text = sections[section_idx]
    
    print(f"\n" + "="*80)
    print(f"ANALYSIS: PATIENT {patient_idx} | SECTION {section_idx}")
    print("="*80)
    print(f"RAW TEXT PREVIEW: {raw_text[:100]}...\n")

    for name, engine_func in MODELS.items():
        sentences = engine_func(raw_text)
        
        print(f"\n>>> {name.upper()} ({len(sentences)} sentences)")
        print("-" * 40)
        
        for i, sent in enumerate(sentences, 1):
            print(f"{i} | {sent}") 
    print("\n" + "*"*80)

In [8]:
test_cases = [
    (1, 2), # Patient 0, Section 0
    (1, 1), # Patient 1, Section 1
]
for p_id, s_id in test_cases:
    compare_patient_output(p_id, s_id)


ANALYSIS: PATIENT 1 | SECTION 2
RAW TEXT PREVIEW: ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ
Βραδυκαρδια(πιθανως φαρμακευτικη).
Διαταση ανιουσης αορτης.
ΦΑΡΜΑΚΕΥΤΙΚΗ ΑΓΩΓΗ ΕΞΟΔΟ...


>>> SPACY (LG) (6 sentences)
----------------------------------------
1 | ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ
Βραδυκαρδια(πιθανως φαρμακευτικη)
2 | .
3 | Διαταση ανιουσης αορτης.
4 | ΦΑΡΜΑΚΕΥΤΙΚΗ ΑΓΩΓΗ ΕΞΟΔΟΥ
Copalia 10 1x1
5 | T4 50 1x1
Trofocard 1x1
Sintrom 4 ως ελαμβανε.
6 | Ληψη μονο Ρυθμονορμ κατα την προσπαθεια αναταξης νεου επεισοδιου κατ οικον.

>>> MEDSPACY (4 sentences)
----------------------------------------
1 | ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ Βραδυκαρδια(πιθανως φαρμακευτικη).
2 | Διαταση ανιουσης αορτης.
3 | ΦΑΡΜΑΚΕΥΤΙΚΗ ΑΓΩΓΗ ΕΞΟΔΟΥ Copalia 10 1x1 T4 50 1x1 Trofocard 1x1 Sintrom 4 ως ελαμβανε.
4 | Ληψη μονο Ρυθμονορμ κατα την προσπαθεια αναταξης νεου επεισοδιου κατ οικον.

>>> STANZA (4 sentences)
----------------------------------------
1 | ΔΙΑΓΝΩΣΗ ΕΞΟΔΟΥ Βραδυκαρδια(πιθανως φαρμακευτικη).
2 | Διαταση ανιουσης αορτης.
3 | ΦΑΡΜΑΚΕΥΤΙΚΗ ΑΓΩΓΗ ΕΞΟΔΟΥ Co